# 07 — Operational Evaluation: 2×2 Comparison

**Author:** Florian Klaver

**Prerequisite:** Notebooks 04–06 must have run. Models must exist in `models/`.

---

## Goal

Apply the trained models to the **full held-out 2020-08-12 scene** (same as the PA2 baseline
in the 5th-semester project) and produce the core 2×2 thesis comparison table:

| | No Postprocessing | With Postprocessing (Morphological+SLIC) |
|---|---|---|
| **S2-only** | F1 = ? | F1 = ? |
| **S2+S1 fusion** | F1 = ? | F1 = ? |

The PA2 baseline (S2-only, old model, no postprocessing) achieved **F1 = 0.27–0.41**.
The postprocessing chapter (notebook 02) demonstrated **F1 = 0.508** with Morphological+SLIC.
This notebook fills in the remaining cells.

### What is computed here

For each of the 4 model+postprocessing combinations:
1. Load the S2 before/after scene (from `data/Sentinel_CH/`)
2. Load the matched S1 before/after images (from `data/Sentinel_S1/`) — fusion only
3. Compute scene-wide features (20 S2 or 26 S2+S1)
4. Apply model → binary prediction raster
5. Optionally apply postprocessing (`morphological+slic`)
6. Evaluate against ground truth → precision, recall, F1

---
## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import rasterio
import joblib
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from evaluation import (
    FEATURE_NAMES, prepare_scene_features, run_prediction, evaluate_and_visualize
)
from sar_features import load_s1_bands, compute_sar_features, align_s1_to_reference
from postprocessing import postprocess_prediction

# --- Paths ---
S2_DIR       = r'..\data\Sentinel_CH'
S1_DIR       = r'..\data\Sentinel_S1'
MASKS_DIR    = r'..\data\ground_truth_masks'
AV_DATA_PATH = r'..\data\amtliche_vermessung_zh\AV_ZH.gpkg'
FB_DATA_PATH = r'..\data\amtliche_vermessung_zh\Feuerwehr_-OGD.gpkg'
MODELS_DIR   = r'..\models'
OUTPUT_DIR   = r'..\data\operational_results'
TEMPORAL_CSV = r'..\data\temporal_matches.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

SAR_FEATURES = ['vv_after', 'vh_after', 'cr_after', 'vv_diff', 'vh_diff', 'cr_diff']
FUSION_FEATURES = FEATURE_NAMES + SAR_FEATURES

print('Setup complete.')

---
## 2. Identify Scene Dates for 2020-08-12

The 2020-08-12 event uses the S2 triplet from `temporal_matches.csv`.
We also need the corresponding S1 dates from `data/s1_date_matches.csv`.

In [ ]:
# Find the 2020-08-12 event in temporal_matches
matches_df = pd.read_csv(TEMPORAL_CSV)
event = matches_df[matches_df['event_date'] == '2020-08-12']

if event.empty:
    # Fallback: find nearest event in August 2020
    matches_df['event_date'] = pd.to_datetime(matches_df['event_date'])
    event = matches_df[
        (matches_df['event_date'].dt.year == 2020) &
        (matches_df['event_date'].dt.month == 8)
    ].iloc[:1]
    print(f"Note: Using nearest August 2020 event: {event.iloc[0]['event_date_str']}")

event = event.iloc[0]
BEFORE_NEAR_S2 = os.path.join(S2_DIR, event['before_near_file'])
AFTER_S2       = os.path.join(S2_DIR, event['after_file'])
GT_MASK_NAME   = event['mask_file']

print(f'Event:           {event["event_date_str"]}')
print(f'Before_near S2:  {event["before_near_file"]}')
print(f'After S2:        {event["after_file"]}')
print(f'Ground truth:    {GT_MASK_NAME}')
print()
print(f'Before_near exists: {os.path.exists(BEFORE_NEAR_S2)}')
print(f'After exists:       {os.path.exists(AFTER_S2)}')
print(f'GT mask exists:     {os.path.exists(os.path.join(MASKS_DIR, GT_MASK_NAME))}')

In [ ]:
# Find matched S1 dates for this event
s1_coverage_path = r'..\data\s1_event_coverage.csv'
if os.path.exists(s1_coverage_path):
    cov_df = pd.read_csv(s1_coverage_path)
    event_cov = cov_df[cov_df['event_date_str'] == event['event_date_str']]
    if not event_cov.empty:
        row = event_cov.iloc[0]
        bn_s1_date = pd.Timestamp(row['bn_s1']).strftime('%Y%m%d') if pd.notna(row['bn_s1']) else None
        af_s1_date = pd.Timestamp(row['af_s1']).strftime('%Y%m%d') if pd.notna(row['af_s1']) else None
        BEFORE_NEAR_S1 = os.path.join(S1_DIR, f'S1_{bn_s1_date}.tif') if bn_s1_date else None
        AFTER_S1       = os.path.join(S1_DIR, f'S1_{af_s1_date}.tif') if af_s1_date else None
        print(f'Before_near S1:  S1_{bn_s1_date}.tif  exists={os.path.exists(BEFORE_NEAR_S1) if BEFORE_NEAR_S1 else False}')
        print(f'After S1:        S1_{af_s1_date}.tif  exists={os.path.exists(AFTER_S1) if AFTER_S1 else False}')
        S1_AVAILABLE = (BEFORE_NEAR_S1 and AFTER_S1 and
                        os.path.exists(BEFORE_NEAR_S1) and os.path.exists(AFTER_S1))
    else:
        S1_AVAILABLE = False
        print('Event not found in s1_event_coverage.csv')
else:
    S1_AVAILABLE = False
    print('s1_event_coverage.csv not found — run notebook 03 first.')

print(f'\nS1 data available for fusion: {S1_AVAILABLE}')

---
## 3. Load Models

In [ ]:
def load_saved_model(path):
    """Load model saved as either a plain joblib or {'model': ..., 'features': ...} dict."""
    obj = joblib.load(path)
    if isinstance(obj, dict):
        return obj['model'], obj.get('features', FEATURE_NAMES)
    return obj, FEATURE_NAMES  # legacy plain model

# S2-only: prefer retrained version, fall back to old PA2 baseline
s2_retrained_path = os.path.join(MODELS_DIR, 's2only_retrained.joblib')
s2_old_path       = os.path.join(MODELS_DIR, 'tuned_svm_hybrid.joblib')

if os.path.exists(s2_retrained_path):
    model_s2, feat_s2 = load_saved_model(s2_retrained_path)
    print(f'S2-only model: s2only_retrained.joblib  ({len(feat_s2)} features)')
elif os.path.exists(s2_old_path):
    model_s2, feat_s2 = load_saved_model(s2_old_path)
    print(f'S2-only model: tuned_svm_hybrid.joblib (PA2 baseline, {len(feat_s2)} features)')
else:
    model_s2 = None
    print('No S2-only model found. Run notebook 06 first.')

fusion_path = os.path.join(MODELS_DIR, 'fusion_best.joblib')
if os.path.exists(fusion_path):
    model_fusion, feat_fusion = load_saved_model(fusion_path)
    print(f'Fusion model:  fusion_best.joblib  ({len(feat_fusion)} features)')
else:
    model_fusion = None
    feat_fusion  = FUSION_FEATURES
    print('No fusion model found. Run notebook 06 first.')

---
## 4. Prepare Scene Features

Compute S2 features for every grassland pixel in the 2020-08-12 scene.
Then append S1 features (for the fusion model).

In [ ]:
# S2 features (shared by both S2-only and fusion)
print('=== Computing S2 scene features ===')
feature_df_s2, valid_mask, meta, H, W = prepare_scene_features(
    BEFORE_NEAR_S2, AFTER_S2, AV_DATA_PATH, FB_DATA_PATH, FEATURE_NAMES
)
print(f'Valid grassland pixels: {valid_mask.sum():,} / {H*W:,}')

In [ ]:
# S1 + Fusion feature DataFrame
if S1_AVAILABLE:
    print('=== Appending S1 SAR features ===')

    bn_vv, bn_vh, _ = align_s1_to_reference(BEFORE_NEAR_S1, AFTER_S2)
    af_vv, af_vh, _ = align_s1_to_reference(AFTER_S1,       AFTER_S2)

    sar_features_scene = compute_sar_features(bn_vv, bn_vh, af_vv, af_vh)

    # Flatten and append SAR features to feature_df for the valid pixels
    feature_df_fusion = feature_df_s2.copy()
    for col, arr in sar_features_scene.items():
        feature_df_fusion[col] = arr.flatten()

    # Update valid_mask: exclude pixels where SAR is also nodata
    sar_stack = np.stack([v.flatten() for v in sar_features_scene.values()])
    sar_nodata = np.isnan(sar_stack).any(axis=0)
    valid_mask_fusion = valid_mask & ~sar_nodata

    print(f'Valid pixels after SAR mask: {valid_mask_fusion.sum():,}')
else:
    feature_df_fusion = None
    valid_mask_fusion = None
    print('S1 data not available — fusion cells in the 2×2 table will be empty.')

---
## 5. Define Postprocessing Function

The postprocessing chapter (notebook 02) established that **Morphological + SLIC** is the
best method (F1 0.274 → 0.508). We apply the same pipeline here.

SLIC requires the RGB AFTER image as context for superpixel segmentation.

In [ ]:
# Load RGB bands from the AFTER S2 image for SLIC
with rasterio.open(AFTER_S2) as src:
    r = np.clip(src.read(3).astype(float) / 2500, 0, 1)
    g = np.clip(src.read(2).astype(float) / 2500, 0, 1)
    b = np.clip(src.read(1).astype(float) / 2500, 0, 1)
scene_rgb = np.dstack([r, g, b]).astype(np.float32)  # shape: (H, W, 3)

def apply_postprocessing(pred_map):
    return postprocess_prediction(
        pred_map,
        method='morphological+slic',
        scene_image=scene_rgb,
        disk_radius=2,
        min_area_pixels=4,
        n_segments=300,
        slic_compactness=0.05,
    )

print('Postprocessing function defined: morphological + SLIC')
print('  → disk_radius=2, min_area_pixels=4, n_segments=300, compactness=0.05')

---
## 6. Run All 4 Cells

Each cell: predict → optionally postprocess → evaluate.

In [ ]:
table_results = {}

def run_cell(label, model, feature_df, valid_mask_px, features, postprocess=False, pp_fn=None):
    """Run one cell of the 2x2 table: predict, optionally postprocess, evaluate."""
    if model is None or feature_df is None:
        print(f'SKIPPED: {label} — model or features not available')
        return None

    print(f'\n{"="*60}\n{label}\n{"="*60}')
    pred_tag  = label.replace(' ', '_').replace('+', 'plus').lower()
    pred_path = os.path.join(OUTPUT_DIR, f'pred_{pred_tag}.tif')

    # Subset features available in the DataFrame
    avail_feats = [f for f in features if f in feature_df.columns]
    run_prediction(model, feature_df[avail_feats], valid_mask_px, H, W, pred_path, meta)

    if postprocess and pp_fn is not None:
        with rasterio.open(pred_path) as src:
            pred_map = src.read(1)
            pp_meta  = src.meta.copy()
        pred_map_pp = pp_fn(pred_map)
        pp_path = pred_path.replace('.tif', '_pp.tif')
        with rasterio.open(pp_path, 'w', **pp_meta) as dst:
            dst.write(pred_map_pp, 1)
        eval_path = pp_path
    else:
        eval_path = pred_path

    metrics = evaluate_and_visualize(
        eval_path, [GT_MASK_NAME], MASKS_DIR, AFTER_S2, OUTPUT_DIR
    )
    table_results[label] = metrics
    return metrics


# ── Cell A: S2-only, no postprocessing ─────────────────────────────────────
run_cell('S2-only | No PP',
         model_s2, feature_df_s2, valid_mask, feat_s2,
         postprocess=False)

In [ ]:
# ── Cell B: S2-only, with postprocessing ────────────────────────────────────
run_cell('S2-only | Morphological+SLIC',
         model_s2, feature_df_s2, valid_mask, feat_s2,
         postprocess=True, pp_fn=apply_postprocessing)

In [ ]:
# ── Cell C: S2+S1 fusion, no postprocessing ──────────────────────────────────
run_cell('S2+S1 Fusion | No PP',
         model_fusion, feature_df_fusion, valid_mask_fusion, feat_fusion,
         postprocess=False)

In [ ]:
# ── Cell D: S2+S1 fusion, with postprocessing ────────────────────────────────
run_cell('S2+S1 Fusion | Morphological+SLIC',
         model_fusion, feature_df_fusion, valid_mask_fusion, feat_fusion,
         postprocess=True, pp_fn=apply_postprocessing)

---
## 7. Core 2×2 Results Table

In [ ]:
# Reconstruct as a formatted 2x2 table
def fmt(key, metric='f1_mowing'):
    m = table_results.get(key)
    if m is None:
        return 'N/A'
    return f'{m[metric]:.3f}  (P={m["precision_mowing"]:.3f}, R={m["recall_mowing"]:.3f})'

print('=== 2×2 Operational Evaluation — 2020-08-12 scene ===')
print(f'{"":30} {"No Postprocessing":35} {"Morphological+SLIC":35}')
print(f'{"─"*100}')
print(f'{"S2-only":30} {fmt("S2-only | No PP"):35} {fmt("S2-only | Morphological+SLIC"):35}')
print(f'{"S2+S1 Fusion":30} {fmt("S2+S1 Fusion | No PP"):35} {fmt("S2+S1 Fusion | Morphological+SLIC"):35}')
print()
print('PA2 baseline (reference): S2-only, no PP, F1 = 0.27–0.41')
print('Postprocessing chapter:   S2-only, Morphological+SLIC, F1 = 0.508')

In [ ]:
# Visual 2x2 heatmap
keys   = ['S2-only | No PP', 'S2-only | Morphological+SLIC',
          'S2+S1 Fusion | No PP', 'S2+S1 Fusion | Morphological+SLIC']
f1_vals = [table_results.get(k, {}).get('f1_mowing', np.nan) for k in keys]

matrix = np.array(f1_vals).reshape(2, 2)
row_labels = ['S2-only', 'S2+S1 Fusion']
col_labels = ['No Postprocessing', 'Morphological+SLIC']

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(matrix, cmap='YlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='F1 Score (mowing class)')
ax.set_xticks([0, 1]); ax.set_xticklabels(col_labels, fontsize=11)
ax.set_yticks([0, 1]); ax.set_yticklabels(row_labels, fontsize=11)
for i in range(2):
    for j in range(2):
        val = matrix[i, j]
        ax.text(j, i, f'{val:.3f}' if not np.isnan(val) else 'N/A',
                ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if val > 0.6 else 'black')
ax.set_title('Operational F1 — 2020-08-12 Scene', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results_2x2_table.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR}/results_2x2_table.png')

---
## Summary

This notebook produces the core empirical contribution of the Bachelor's thesis:
a 2×2 operational F1 table comparing Sentinel-2-only vs. S2+S1 fusion, with and
without Morphological+SLIC postprocessing, on the held-out 2020-08-12 airport scene.

**Outputs in `data/operational_results/`:**
- `pred_*.tif` — prediction rasters for each of the 4 configurations
- `eval_map_*.png` — spatial difference maps (TP/FP/FN) + confusion matrices
- `results_2x2_table.png` — heatmap of the 2×2 F1 results